# <div align="center"> Machine Learning And Econometrics  </div>
## <div align="center"> ECO 4971/6971  </div>
#### <div align="center">Class 8B - Classification (ISLR Ch. 4)</div>
<div align="center"> Jonathan Holmes, (he/him)</div>

# Overview

- In this class we continue **classification** using the `Credit` dataset from *Introduction to Statistical Learning* (ISLR).
- We will focus on several **Chapter 4** concepts:
    1. **Multinomial logistic regression**
    2. **Generative models for classification**
    3. **Linear discriminant analysis (LDA)**
    4. **Quadratic discriminant analysis (QDA)**
    5. **Comparing different classification models**
- We will construct a simple classification problem based on **credit card balances** and compare methods.

In [ ]:
# Imports
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, accuracy_score

sns.set(style="whitegrid")

%matplotlib inline

In [ ]:
# Load Credit data from ISLR (CSV bundled with this class)

credit = pd.read_csv("https://www.dropbox.com/scl/fi/dlvn95r99i6da5oc7ha82/Credit.csv?rlkey=rbnd58yh0s06lu8hmyj3esjzi&dl=1")
credit.head()

## Constructing a classification problem

- The original `Credit` dataset is not a binary default dataset.
- To practice Chapter 4 methods, we will create two new outcomes:
    - **`HighBalance`**: indicator for whether a customer has a high credit card balance
    - **`BalanceLevel`**: a **3-category** version of balance (low / medium / high)
- These will let us illustrate **binary** and **multinomial** classification on the same data.

In [ ]:
# Create binary and multiclass outcomes based on Balance

# HighBalance = 1 if balance is in the top quartile
threshold = credit["Balance"].quantile(0.75)
credit["HighBalance"] = (credit["Balance"] >= threshold).astype(int)

# BalanceLevel: 0 = low, 1 = medium, 2 = high (terciles)
credit["BalanceLevel"] = pd.qcut(credit["Balance"], q=3, labels=[0, 1, 2]).astype(int)

credit[["Balance", "HighBalance", "BalanceLevel"]].head()

In [ ]:
# Class proportions

print("HighBalance (binary):")
print(credit["HighBalance"].value_counts(normalize=True))
print("\nBalanceLevel (3 classes):")
print(credit["BalanceLevel"].value_counts(normalize=True))

In [ ]:
# Quick visualization of Balance by Student status

plt.figure(figsize=(6, 4))
sns.boxplot(data=credit, x="Student", y="Balance")
plt.title("Credit card balance by student status")
plt.show()

# 1. Multinomial logistic regression

- In Chapter 4, ISLR introduces **logistic regression** for binary outcomes.
- Multinomial logistic regression generalizes this idea to **3 or more categories**.
- Suppose there are \(K\) outcome categories and let category \(K\) be the **baseline**.
  For predictors \(x\), the model is
$$ \Pr(Y = k \mid X = x)
   = \frac{\exp(\beta_{0k} + \beta_k^\top x)}{1 + \sum_{j=1}^{K-1} \exp(\beta_{0j} + \beta_j^\top x)}, \quad k = 1,\dots,K-1, $$
$$ \Pr(Y = K \mid X = x)
   = \frac{1}{1 + \sum_{j=1}^{K-1} \exp(\beta_{0j} + \beta_j^\top x)}. $$
- Equivalently, the **log-odds** of being in class \(k\) versus the baseline are linear in \(x\):

$$ \log[\Pr(Y=k\mid X=x)/\Pr(Y=K\mid X=x)] = \beta_{0k} + \beta_k^\top x$$ 
- The parameters $\beta_{0k}, \beta_k$ are estimated by **maximum likelihood**, and
  for prediction we assign each observation to the class with the largest
  predicted probability.
- In our application we model `BalanceLevel` (low / medium / high) as a function of:
    - `Income`
    - `Limit`
    - `Student` (indicator)
- We use scikit-learn's `LogisticRegression` with `multi_class="multinomial"`. 

In [ ]:
# Prepare data for multinomial logistic regression
X_multi = credit[["Income", "Limit", "Student"]].copy()

# One-hot encode Student: assume values "Yes"/"No"
X_multi = pd.get_dummies(X_multi, columns=["Student"], drop_first=True)
y_multi = credit["BalanceLevel"]

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multi, y_multi, test_size=0.3, random_state=123
)

scaler_m = StandardScaler()
X_train_m_scaled = scaler_m.fit_transform(X_train_m)
X_test_m_scaled = scaler_m.transform(X_test_m)

print("test")

multi_logit = LogisticRegression(solver="lbfgs", max_iter=1000, penalty=None)
multi_logit.fit(X_train_m_scaled, y_train_m)


print("Classes:", multi_logit.classes_)
print("Intercepts:", multi_logit.intercept_)
print("Coefficients:\n", multi_logit.coef_)

# Ways of evaluating a classifier

- **Confusion matrix**
    - A table that compares **predicted** vs **actual** class labels.
    - For a binary problem we usually write:
        - **TP (true positives)**: predicted 1, actual 1
        - **FP (false positives)**: predicted 1, actual 0
        - **FN (false negatives)**: predicted 0, actual 1
        - **TN (true negatives)**: predicted 0, actual 0

- **Precision**
    - Among all observations we **predicted as positive**, the fraction that are actually positive.
    - Formula (binary case):
      $$ \text{Precision} = \frac{TP}{TP + FP}. $$

- **Recall (Sensitivity / True positive rate)**
    - Among all observations that are **actually positive**, the fraction that we correctly predict as positive.
    - Formula:
      $$ \text{Recall} = \frac{TP}{TP + FN}. $$

- **F1 score**
    - A single summary of performance that balances precision and recall.
    - Defined as the **harmonic mean** of precision and recall:
      $$ \text{F1} = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}. $$
    - High F1 requires both precision and recall to be reasonably large (penalizes models that are good at only one of them).

In [ ]:
# In-sample accuracy and confusion matrix for multinomial logit

y_pred_m = multi_logit.predict(X_test_m_scaled)

print("Accuracy (multinomial logit):", accuracy_score(y_test_m, y_pred_m))
print("\nClassification report:\n", classification_report(y_test_m, y_pred_m))

ConfusionMatrixDisplay.from_predictions(y_test_m, y_pred_m)
plt.title("Multinomial logistic regression: BalanceLevel")
plt.show()

# 2. Generative models for classification

- Logistic regression directly models
$$ Pr(Y = k \mid X = x) $$
  (a **discriminative** model).
- In ISLR, a **generative** model instead specifies
$$ f_k(x) = p(X = x \mid Y = k) $$
  for each class \(k\), together with class prior probabilities \(\pi_k = Pr(Y = k)\).
- Using Bayes' rule, we then obtain
$$ Pr(Y = k \mid X = x) = \frac{\pi_k f_k(x)}{\sum_{\ell} \pi_\ell f_\ell(x)}. $$
- **LDA** and **QDA** are examples of generative classification models that assume
  multivariate normal densities for \(f_k(x)\).

# 3. Linear Discriminant Analysis (LDA)

- LDA assumes that, conditional on class \(Y=k\), the predictors are multivariate
  normal with a **shared covariance matrix** across classes.
- This implies that the log posterior odds are **linear** functions of \(x\).
- We first apply LDA to a **binary** problem: predicting `HighBalance` using
  `Income` and `Limit`.
- Then we will compare to logistic regression and QDA.

In [ ]:
# Data for binary classification: HighBalance

X_bin = credit[["Income", "Limit"]]
y_bin = credit["HighBalance"]

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_bin, y_bin, test_size=0.3, random_state=123
)

scaler_b = StandardScaler()
X_train_b_scaled = scaler_b.fit_transform(X_train_b)
X_test_b_scaled = scaler_b.transform(X_test_b)

# Logistic regression (binary) for comparison
logit_bin = LogisticRegression(solver="lbfgs")
logit_bin.fit(X_train_b_scaled, y_train_b)

# LDA
lda = LinearDiscriminantAnalysis()
lda.fit(X_train_b_scaled, y_train_b)

for name, model in [("Logit", logit_bin), ("LDA", lda)]:
    y_pred = model.predict(X_test_b_scaled)
    acc = accuracy_score(y_test_b, y_pred)
    print(f"{name} accuracy: {acc:.3f}")

In [ ]:
# Confusion matrices for logistic regression and LDA

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, (name, model) in zip(axes, [("Logit", logit_bin), ("LDA", lda)]):
    ConfusionMatrixDisplay.from_predictions(
        y_test_b,
        model.predict(X_test_b_scaled),
        display_labels=["Low", "High"],
        ax=ax,
        colorbar=False,
    )
    ax.set_title(f"{name} (HighBalance)")

plt.tight_layout()
plt.show()

# 4. Quadratic Discriminant Analysis (QDA)

- QDA uses the same generative idea as LDA but allows each class to have its own
  covariance matrix.
- This leads to **quadratic decision boundaries**.
- QDA can capture more flexible relationships but may overfit, especially with
  limited data or many predictors.
- We fit QDA to the same `HighBalance` problem.

In [ ]:
# QDA for HighBalance

qda = QuadraticDiscriminantAnalysis()
qda.fit(X_train_b_scaled, y_train_b)

y_pred_qda = qda.predict(X_test_b_scaled)
acc_qda = accuracy_score(y_test_b, y_pred_qda)

print(f"QDA accuracy: {acc_qda:.3f}")

ConfusionMatrixDisplay.from_predictions(
    y_test_b,
    y_pred_qda,
    display_labels=["Low", "High"],
)
plt.title("QDA (HighBalance)")
plt.show()

# 5. Comparing models

- We now compare several models on the same binary classification task
  (`HighBalance`):
    - Logistic regression
    - LDA
    - QDA
    - Gaussian Naive Bayes (another generative model)
- We report out-of-sample accuracy on a common train/test split.

In [ ]:
# Model comparison on HighBalance

models = {
    "Logit": LogisticRegression(solver="lbfgs"),
    "LDA": LinearDiscriminantAnalysis(),
    "QDA": QuadraticDiscriminantAnalysis(),
    "NaiveBayes": GaussianNB(),
}

results = []

for name, model in models.items():
    model.fit(X_train_b_scaled, y_train_b)
    y_pred = model.predict(X_test_b_scaled)
    acc = accuracy_score(y_test_b, y_pred)
    results.append({"model": name, "accuracy": acc})

results_df = pd.DataFrame(results).sort_values("accuracy", ascending=False)
results_df

## Discussion

- **Logistic regression vs. LDA**
    - Both produce linear decision boundaries in the predictors.
    - LDA relies on a multivariate normal assumption; logistic regression does not.
- **QDA**
    - More flexible (quadratic boundaries) but can have higher variance.
- **Generative vs. discriminative**
    - Generative models (LDA, QDA, Naive Bayes) estimate \(f_k(x)\) and priors.
    - Discriminative models (logit) directly estimate \(Pr(Y=k \mid X=x)\).
- In practice, we often compare several models using **out-of-sample performance**
  and choose the one that balances **accuracy**, **interpretability**, and
  **assumptions**.